# Checkpoint Three: Cleaning Data

Now you are ready to clean your data. Before starting coding, provide the link to your dataset below.

My dataset:https://www.kaggle.com/datasets/jessemostipak/hotel-booking-demand

Import the necessary libraries and create your dataframe(s).

In [2]:
import pandas as pd
import numpy as np

data = pd.read_csv("hotel_bookings.csv")
df = pd.DataFrame(data)
# getting df overview 
df.info()
# getting first few rows to use for reference when handling values
df.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 119390 entries, 0 to 119389
Data columns (total 32 columns):
 #   Column                          Non-Null Count   Dtype  
---  ------                          --------------   -----  
 0   hotel                           119390 non-null  object 
 1   is_canceled                     119390 non-null  int64  
 2   lead_time                       119390 non-null  int64  
 3   arrival_date_year               119390 non-null  int64  
 4   arrival_date_month              119390 non-null  object 
 5   arrival_date_week_number        119390 non-null  int64  
 6   arrival_date_day_of_month       119390 non-null  int64  
 7   stays_in_weekend_nights         119390 non-null  int64  
 8   stays_in_week_nights            119390 non-null  int64  
 9   adults                          119390 non-null  int64  
 10  children                        119386 non-null  float64
 11  babies                          119390 non-null  int64  
 12  meal            

,hotel,is_canceled,lead_time,arrival_date_year,arrival_date_month,arrival_date_week_number,arrival_date_day_of_month,stays_in_weekend_nights,stays_in_week_nights,adults,...,deposit_type,agent,company,days_in_waiting_list,customer_type,adr,required_car_parking_spaces,total_of_special_requests,reservation_status,reservation_status_date
0,Resort Hotel,0,342,2015,July,27,1,0,0,2,...,No Deposit,NaN,NaN,0,Transient,0.0,0,0,Check-Out,2015-07-01
1,Resort Hotel,0,737,2015,July,27,1,0,0,2,...,No Deposit,NaN,NaN,0,Transient,0.0,0,0,Check-Out,2015-07-01
2,Resort Hotel,0,7,2015,July,27,1,0,1,1,...,No Deposit,NaN,NaN,0,Transient,75.0,0,0,Check-Out,2015-07-02
3,Resort Hotel,0,13,2015,July,27,1,0,1,1,...,No Deposit,304.0,NaN,0,Transient,75.0,0,0,Check-Out,2015-07-02
4,Resort Hotel,0,14,2015,July,27,1,0,2,2,...,No Deposit,240.0,NaN,0,Transient,98.0,0,1,Check-Out,2015-07-03


In [3]:
#converting to useable date for later use
df['reservation_status_date']=pd.to_datetime(df['reservation_status_date'])

## Missing Data

Test your dataset for missing data and handle it as needed. Make notes in the form of code comments as to your thought process.

In [4]:
# getting null count 
df.isnull().sum()

hotel                                  0
is_canceled                            0
lead_time                              0
arrival_date_year                      0
arrival_date_month                     0
arrival_date_week_number               0
arrival_date_day_of_month              0
stays_in_weekend_nights                0
stays_in_week_nights                   0
adults                                 0
children                               4
babies                                 0
meal                                   0
country                              488
market_segment                         0
distribution_channel                   0
is_repeated_guest                      0
previous_cancellations                 0
previous_bookings_not_canceled         0
reserved_room_type                     0
assigned_room_type                     0
booking_changes                        0
deposit_type                           0
agent                              16340
company         

In [5]:
# only 488 values to handle and every booking must have come from somewhere so updating to most frequent value
df['country'].fillna(df['country'].mode()[0], inplace=True)

# only 4 values replacing with 0
df['children'].fillna(0, inplace=True)

# not every booking comes via an agent or is booked by a company; therefore i'm making them "none" in order to distinguish that they were not mistaken nulls
df['company'].fillna('None', inplace=True)
df['agent'].fillna('None', inplace=True)

# all nulls handled
df.isnull().sum()


/var/folders/q2/c8thy5d901lc5m5ygppj1ss40000gn/T/ipykernel_45472/2204615282.py:2: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df['country'].fillna(df['country'].mode()[0], inplace=True)
/var/folders/q2/c8thy5d901lc5m5ygppj1ss40000gn/T/ipykernel_45472/2204615282.py:5: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting 

hotel                             0
is_canceled                       0
lead_time                         0
arrival_date_year                 0
arrival_date_month                0
arrival_date_week_number          0
arrival_date_day_of_month         0
stays_in_weekend_nights           0
stays_in_week_nights              0
adults                            0
children                          0
babies                            0
meal                              0
country                           0
market_segment                    0
distribution_channel              0
is_repeated_guest                 0
previous_cancellations            0
previous_bookings_not_canceled    0
reserved_room_type                0
assigned_room_type                0
booking_changes                   0
deposit_type                      0
agent                             0
company                           0
days_in_waiting_list              0
customer_type                     0
adr                         

## Irregular Data

Detect outliers in your dataset and handle them as needed. Use code comments to make notes about your thought process.

In [6]:
# looking for red flags with stats
df.describe()
# adding red flag columns to list for iteration
outliers = ['adults','children', 'adr','lead_time','required_car_parking_spaces']

for i in outliers:
    # calculate quartiles and getting IQR (only removing upper and lower 10% to cut worst offenders)
    q1 = df[i].quantile(0.10)
    q3 = df[i].quantile(0.90)
    iqr = q3 - q1
    
    # setting bounds for data replacement
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    
    #anything lower than the lower bound gets replaced with the lower bound itself
    df[i] = np.where(df[i] < lower, lower, df[i])

    #anything higher than the upper bound gets replaced with the upper bound itself
    df[i] = np.where(df[i] > upper, upper, df[i])

In [7]:
#rounding because it doesnt make sense for these data points to be floats. ie you cant have .5 adults also converting to int dtypes
#reusing same list as above
for i in outliers:
    df[i] = df[i].round().astype(int)

## Unnecessary Data

Look for the different types of unnecessary data in your dataset and address it as needed. Make sure to use code comments to illustrate your thought process.

In [8]:
# Dropping columns as they are unhelpful
df.drop(columns=['stays_in_week_nights'], inplace=True)
df.drop(columns=['stays_in_weekend_nights'], inplace=True)

#dropping column for being a duplicate of distribution channel
df.drop(columns=['market_segment'], inplace=True)
#dropping column for being a duplicate of children
df.drop(columns=['babies'], inplace=True)
#dropping for being duplicate of reserved room type and reserved room type tells us more about the customers preferences.
df.drop(columns=['reserved_room_type'], inplace=True)

#dropping the following columns for having so few data points that it is statistically insignificant
df.drop(columns=['is_repeated_guest'], inplace=True)
df.drop(columns=['previous_cancellations'], inplace=True)
df.drop(columns=['previous_bookings_not_canceled'], inplace=True)

#confirming changes
df.head()

,hotel,is_canceled,lead_time,arrival_date_year,arrival_date_month,arrival_date_week_number,arrival_date_day_of_month,adults,children,meal,...,deposit_type,agent,company,days_in_waiting_list,customer_type,adr,required_car_parking_spaces,total_of_special_requests,reservation_status,reservation_status_date
0,Resort Hotel,0,342,2015,July,27,1,2,0,BB,...,No Deposit,None,None,0,Transient,0,0,0,Check-Out,2015-07-01
1,Resort Hotel,0,658,2015,July,27,1,2,0,BB,...,No Deposit,None,None,0,Transient,0,0,0,Check-Out,2015-07-01
2,Resort Hotel,0,7,2015,July,27,1,1,0,BB,...,No Deposit,None,None,0,Transient,75,0,0,Check-Out,2015-07-02
3,Resort Hotel,0,13,2015,July,27,1,1,0,BB,...,No Deposit,304.0,None,0,Transient,75,0,0,Check-Out,2015-07-02
4,Resort Hotel,0,14,2015,July,27,1,2,0,BB,...,No Deposit,240.0,None,0,Transient,98,0,1,Check-Out,2015-07-03


## Inconsistent Data

Check for inconsistent data and address any that arises. As always, use code comments to illustrate your thought process.

In [9]:
# the adr or "average daily rate" doesnt make sense as a 0 therefore I'm filtering those records
df = df[(df['adr'] > 0)]
#after filtering both children and babies columns are 100% 0 therefore removing column entirely
df.drop(columns=['children'], inplace=True)
#same as above
df.drop(columns=['required_car_parking_spaces'], inplace=True)

#checking changes
df.head()

,hotel,is_canceled,lead_time,arrival_date_year,arrival_date_month,arrival_date_week_number,arrival_date_day_of_month,adults,meal,country,...,booking_changes,deposit_type,agent,company,days_in_waiting_list,customer_type,adr,total_of_special_requests,reservation_status,reservation_status_date
2,Resort Hotel,0,7,2015,July,27,1,1,BB,GBR,...,0,No Deposit,None,None,0,Transient,75,0,Check-Out,2015-07-02
3,Resort Hotel,0,13,2015,July,27,1,1,BB,GBR,...,0,No Deposit,304.0,None,0,Transient,75,0,Check-Out,2015-07-02
4,Resort Hotel,0,14,2015,July,27,1,2,BB,GBR,...,0,No Deposit,240.0,None,0,Transient,98,1,Check-Out,2015-07-03
5,Resort Hotel,0,14,2015,July,27,1,2,BB,GBR,...,0,No Deposit,240.0,None,0,Transient,98,1,Check-Out,2015-07-03
6,Resort Hotel,0,0,2015,July,27,1,2,BB,PRT,...,0,No Deposit,None,None,0,Transient,107,0,Check-Out,2015-07-03


## Summarize Your Results

Make note of your answers to the following questions.

1. Did you find all four types of dirty data in your dataset?
2. Did the process of cleaning your data give you new insights into your dataset?
3. Is there anything you would like to make note of when it comes to manipulating the data and making visualizations?

ANSWER 1: YES! duplicate columns, missing data, outliers like 50 children, and inconsistent data.

ANSWER 2: Definitely, I took a significantly longer time evaluating each columns purpose and potential use.

ANSWER 3: The EDA I performed would have drastically changed had I cleaned the data more thoroughly.

In [10]:
df.to_csv('clean_data.csv', index=False)